$$
\text{Sign}(x) \;=\; \text{Sign}_{0,1,0}(x) \;=\; \begin{cases} 1 & x > 0 \\ 0 & x = 0 \\ -1 & x < 0 \end{cases}
$$


$$
\text{Sign}_{a,b,c}(x) \;=\; a + (b-a)\,\text{Sign}(x - c) \;=\; \begin{cases} b & x > c \\ \dfrac{a+b}{2} & x = c \\ a & x < c \end{cases}
$$

$$
\begin{aligned}
(x == a) &:= (x > a - \epsilon) \cdot (x < a + \epsilon) \\
(x_i == 0) &:= (x_i \geq -0.01) \cdot (x_i \leq 0.01) \\[6pt]
x_i = 0.005: \quad &(0.005 \geq -0.01) \cdot (0.005 \leq 0.01) = 1 \cdot 1 = 1 \\
x_i = 0.5: \quad &(0.5 \geq -0.01) \cdot (0.5 \leq 0.01) = 1 \cdot 0 = 0
\end{aligned}
$$

## Tanh Sign Convergence 

### Plotted Approxiation Dataset Output

In [8]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Load data (long format: one row per (k, x); x in [-2, 2], 0.001 step)
df = pd.read_csv("tests/sign_tanh_convergence_results.csv")

# The 9-term Taylor series for tanh only converges for |k*x| < pi/2, so each
# approximation diverges (up to ~1e28 for k=64) once x leaves |x| < pi/(2k).
# Mask the divergent tails to NaN so each line draws only over its valid
# region; Plotly breaks the line at NaNs, leaving clean central curves.
MASK_THRESHOLD = 2.0
df["tanh_approximation"] = df["tanh_approximation"].where(
    df["tanh_approximation"].abs() <= MASK_THRESHOLD, np.nan
)

# Treat k as a discrete category so each k gets its own colored line/legend.
df["k"] = df["k"].astype(str)


def plot_tanh_sign(df, k_list, title, x_range=(-2, 2)):
    """One masked approximation line per k in k_list, plus the True Sign step."""
    order = [str(k) for k in k_list]
    sub = df[df["k"].isin(order)]

    fig = px.line(
        sub,
        x="x_value",
        y="tanh_approximation",
        color="k",
        category_orders={"k": order},
        title=title,
        labels={
            "x_value": "Plaintext Input",
            "tanh_approximation": "Calculated Value",
            "k": "k",
        },
    )

    # Overlay the true sign function once (identical across k) as a reference.
    true = sub[sub["k"] == order[0]]
    fig.add_trace(
        go.Scatter(
            x=true["x_value"],
            y=true["sign_real"],
            mode="lines",
            name="True Sign",
            line=dict(color="black", dash="dash", width=2),
        )
    )

    fig.update_xaxes(range=list(x_range))
    fig.update_yaxes(range=[-0.5, 1.5])
    return fig


# Graph 1: low k -- wide valid windows, so show the full x-range.
plot_tanh_sign(
    df,
    [1, 2, 4],
    "Tanh Sign Approximation (n_terms = 9) vs True Sign — low k (1, 2, 4)",
    x_range=(-2, 2),
).show()

In [9]:
# Graph 2: high k -- narrow valid windows (k=64 is only +/-0.03), so zoom in
# on x near 0 to make the steep transitions readable.
plot_tanh_sign(
    df,
    [8, 16, 32, 64],
    "Tanh Sign Approximation (n_terms = 9) vs True Sign — high k (8, 16, 32, 64)",
    x_range=(-0.3, 0.3),
).show()

## Convergence Radius Analysis

In [18]:
# Per-k accuracy of the tanh sign approximation vs. the true sign function.

import pandas as pd
import numpy as np

df = pd.read_csv("tests/sign_tanh_convergence_results.csv")

# Error tolerance for the "accurate" sub-window. The approximation's accuracy
# ceiling is ~8.3% (its best value is 0.917, never reaching 1), so a 10% band
# is just wide enough to admit a usable window near the convergence radius.
ACC_TOL = 0.10


def accuracy_within_window(sub_k):
    """For a single k: locate the empirical convergence window, measure how far
    the approximation is from the true sign inside it, and find the sub-range
    on each side where that error stays within ACC_TOL.

    The window edge is the curve's turning point, searched only within the
    theoretical radius pi/(2k) where the 9-term Taylor series converges -- so no
    arbitrary value mask is needed. Error is |approx - sign_real| (distance from
    1 for x>0, from 0 for x<0)."""
    k = int(sub_k['k'].iloc[0])
    radius = np.pi / (2 * k)

    left  = sub_k[(sub_k['x_value'] < 0) & (sub_k['x_value'] >= -radius)]
    right = sub_k[(sub_k['x_value'] > 0) & (sub_k['x_value'] <=  radius)]

    # Empirical radius on each side = the turning point inside the convergent zone.
    rhs_radius = right.loc[right['tanh_approximation'].idxmax(), 'x_value']
    lhs_radius = left.loc[left['tanh_approximation'].idxmin(),  'x_value']

    rhs_win = right[right['x_value'] <= rhs_radius].copy()
    lhs_win = left[left['x_value']  >= lhs_radius].copy()

    rhs_win['err'] = (rhs_win['tanh_approximation'] - rhs_win['sign_real']).abs()
    lhs_win['err'] = (lhs_win['tanh_approximation'] - lhs_win['sign_real']).abs()

    # Sub-range on each side where error <= ACC_TOL (empty -> NaN bounds).
    rhs_ok = rhs_win.loc[rhs_win['err'] <= ACC_TOL, 'x_value']
    lhs_ok = lhs_win.loc[lhs_win['err'] <= ACC_TOL, 'x_value']
    rhs_lo, rhs_hi = (rhs_ok.min(), rhs_ok.max()) if len(rhs_ok) else (np.nan, np.nan)
    lhs_lo, lhs_hi = (lhs_ok.min(), lhs_ok.max()) if len(lhs_ok) else (np.nan, np.nan)

    return {
        'k': k,
        'lhs_radius':  lhs_radius,
        'rhs_radius':  rhs_radius,
        'lhs_max_%':   lhs_win['err'].max()  * 100,
        'rhs_max_%':   rhs_win['err'].max()  * 100,
        # range where error stays within ACC_TOL
        'lhs_<=tol':   (lhs_lo, lhs_hi),
        'rhs_<=tol':   (rhs_lo, rhs_hi),
    }


rows   = [accuracy_within_window(g) for _, g in df.groupby('k')]
report = pd.DataFrame(rows).sort_values('k').reset_index(drop=True)

pd.set_option('display.float_format', lambda v: f"{v:.3f}")
print(f"Tanh sign accuracy vs. true sign, per k   (error % = 100 * |approx - sign|)")
print(f"Within-tolerance ranges use ACC_TOL = {ACC_TOL:.0%}")
print("=" * 78)
print(report.to_string(index=False))

print("\nPrintable statements:")
print("-" * 78)
for r in rows:
    (llo, lhi), (rlo, rhi) = r['lhs_<=tol'], r['rhs_<=tol']
    print(f"  k={r['k']:>2}: within {ACC_TOL:.0%} of true sign on "
          f"x in [{llo:.3f}, {lhi:.3f}] (left) and "
          f"[{rlo:.3f}, {rhi:.3f}] (right).")


Tanh sign accuracy vs. true sign, per k   (error % = 100 * |approx - sign|)
Within-tolerance ranges use ACC_TOL = 10%
 k  lhs_radius  rhs_radius  lhs_max_%  rhs_max_%        lhs_<=tol      rhs_<=tol
 1      -1.272       1.272     49.950     49.950 (-1.272, -1.105) (1.105, 1.272)
 2      -0.636       0.636     49.900     49.900 (-0.636, -0.553) (0.553, 0.636)
 4      -0.318       0.318     49.800     49.800 (-0.318, -0.277) (0.277, 0.318)
 8      -0.159       0.159     49.600     49.600 (-0.159, -0.139) (0.139, 0.159)
16      -0.079       0.079     49.200     49.200  (-0.079, -0.07)  (0.07, 0.079)
32      -0.040       0.040     48.401     48.401  (-0.04, -0.035)  (0.035, 0.04)
64      -0.020       0.020     46.804     46.804  (-0.02, -0.018)  (0.018, 0.02)

Printable statements:
------------------------------------------------------------------------------
  k= 1: within 10% of true sign on x in [-1.272, -1.105] (left) and [1.105, 1.272] (right).
  k= 2: within 10% of true sign on x in 

In [17]:
# Visualize the k=1 tanh sign approximation with its <=10% accuracy ring shaded.
# Reuses accuracy_within_window() and ACC_TOL from the cell above.

import plotly.graph_objects as go

K = 1
sub = df[df['k'] == K].sort_values('x_value')
stats = accuracy_within_window(sub)
(llo, lhi) = stats['lhs_<=tol']   # left  band: x in [llo, lhi]
(rlo, rhi) = stats['rhs_<=tol']   # right band: x in [rlo, rhi]

# Mask the divergent Taylor tail so the line only draws over its valid region.
plot = sub.copy()
plot['tanh_approximation'] = plot['tanh_approximation'].where(
    plot['tanh_approximation'].abs() <= 2, np.nan
)

fig = go.Figure()

# Shaded <=10% accuracy rings (left and right), drawn first so curves sit on top.
for lo, hi in [(llo, lhi), (rlo, rhi)]:
    fig.add_vrect(
        x0=lo, x1=hi,
        fillcolor="green", opacity=0.18, line_width=0,
        annotation_text=f"≤{ACC_TOL:.0%} error",
        annotation_position="top left",
    )

# Approximation curve and the true sign reference.
fig.add_trace(go.Scatter(
    x=plot['x_value'], y=plot['tanh_approximation'],
    mode="lines", name=f"Tanh approx (k={K}, n_terms=9)",
    line=dict(color="royalblue", width=2),
))
fig.add_trace(go.Scatter(
    x=sub['x_value'], y=sub['sign_real'],
    mode="lines", name="True Sign",
    line=dict(color="black", dash="dash", width=2),
))

fig.update_xaxes(title="Plaintext Input", range=[-2, 2])
fig.update_yaxes(title="Calculated Value", range=[-0.5, 1.5])
fig.update_layout(
    title=f"Tanh Sign Approximation (k={K}) with ≤{ACC_TOL:.0%} Accuracy Ring Shaded",
)
fig.show()
